In [17]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import json
import os
import numpy as np
from tqdm import tqdm

In [18]:
TEST_DIR = "PlantDoc-Dataset/test/"
CLASSMAP_PATH = 'classmap.json'
MODEL_CHECKPOINT = 'runs/mixed_with_fda/run1/best_mixed_fda.pt'
IMG_SIZE = 224
BATCH_SIZE = 32
DEVICE = 'cpu'

In [19]:
class RobustMixedDataset(Dataset):
    def __init__(self, pv_dir, pd_dir, classmap_path, 
                 transform=None):
        self.transform = transform
        self.samples = [] 
        
        with open(classmap_path, 'r') as f:
            cm = json.load(f)
        
        pv_map = cm["plantvillage_to_unified"]
        pd_map = cm["plantdoc_to_unified"]
        
        pv_classes = set(pv_map.values())
        pd_classes = set(pd_map.values())
        common_classes = sorted(list(pv_classes.intersection(pd_classes)))
        
        self.class_to_idx = {name: i for i, name in enumerate(common_classes)}
        self.classes = common_classes # List of class names
        
        self._collect(pd_dir, pd_map)

    def _collect(self, root, mapping):
        if not os.path.exists(root): return
        for folder in os.listdir(root):
            path = os.path.join(root, folder)
            if not os.path.isdir(path): continue
            
            # Map folder name to unified class name
            unified_name = mapping.get(folder)
            
            if unified_name in self.class_to_idx:
                label_idx = self.class_to_idx[unified_name]
                for f in os.listdir(path):
                    if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                        full_path = os.path.join(path, f)
                        self.samples.append((full_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

In [20]:
def evaluate_topk(model, loader, device, k=5):
    model.eval()
    correct_1 = 0
    correct_k = 0
    total = 0
    
    print(f"Evaluating on {DEVICE}...")
    
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Testing"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Get model outputs
            outputs = model(inputs)
            
            # Get Top-K predictions
            _, pred_indices = outputs.topk(k, dim=1, largest=True, sorted=True)
            
            pred_indices = pred_indices.t()
            
            target_expanded = labels.view(1, -1).expand_as(pred_indices)
            correct = pred_indices.eq(target_expanded)
            
            # Top-1 
            correct_1 += correct[:1].reshape(-1).float().sum(0, keepdim=True).item()
            
            # Top-K 
            correct_k += correct[:k].reshape(-1).float().sum(0, keepdim=True).item()
            
            total += labels.size(0)
            
    acc_1 = 100.0 * correct_1 / total
    acc_k = 100.0 * correct_k / total
    
    return acc_1, acc_k, total

In [21]:
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Loading Test Dataset...")
test_dataset = RobustMixedDataset(
    pv_dir="", 
    pd_dir=TEST_DIR, 
    classmap_path=CLASSMAP_PATH, 
    transform=val_transform
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Found {len(test_dataset)} test images.")

print("Loading Model...")
checkpoint = torch.load(MODEL_CHECKPOINT, map_location=DEVICE)
num_classes = len(checkpoint['class_to_idx']) # Use the saved class count

model = models.mobilenet_v3_small(weights=None) # No init weights needed, we load ours
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
model.load_state_dict(checkpoint['model'])
model.to(DEVICE)

top1, top5, total_samples = evaluate_topk(model, test_loader, DEVICE, k=5)

print("\n" + "="*30)
print(f"EVALUATION RESULTS ({total_samples} images)")
print("="*30)
print(f"Top-1 Accuracy: {top1:.2f}%")
print(f"Top-5 Accuracy: {top5:.2f}%")
print("="*30)

Loading Test Dataset...
Found 236 test images.
Loading Model...
Evaluating on cpu...


Testing: 100%|██████████| 8/8 [00:14<00:00,  1.79s/it]


EVALUATION RESULTS (236 images)
Top-1 Accuracy: 60.17%
Top-5 Accuracy: 84.75%
